In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


In [2]:
# Cell 1: Imports and setup
import warnings
import os
import sys
import math
import time
import datetime

import numpy as np
import dill
import scipy.stats
from IPython.display import display, Markdown as md

# Suppress JupyterViz warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*JupyterViz: `use_state` found within a nested function.*")
warnings.filterwarnings("ignore", module="solara.*")
warnings.filterwarnings("ignore", category=UserWarning)


# Your custom modules
from resistance_model import *



In [3]:
from IPython.display import Markdown as md
import numpy as np
import dill
import os
import scipy.stats

select = 0
displayMode = ["singleRunVisualize", "batchRun", "loadSession", "loadSensAnalysis"][select]

override_mesa_shapes() #built-in mesa draw doesn't have shapes - this overwrites with custom shape-including one


In [4]:
def agent_portrayal(agent):
    """Return color, size, and marker style for a given agent type."""

    # defaults
    color = "gray"
    marker = "."
    size = 30
    # ---------------- CITIZENS ----------------
    if agent.type == ATypes.CITIZEN:
        marker = "o"
        size = 30

        if agent.jailTerm:
            color = "white"                 # REMOVED from system
        elif agent.inProtest:
            color = "cyan"                  # ACTION
        elif agent.activeNV:
            color = "mediumaquamarine"              # ORGANIZED / MOBILIZED
        elif agent.grievanceLevel == "high":
            color = "lightblue"      # LATENT PRESSURE
        else:
            color = "lightcyan"             # BACKGROUND

    # ---------------- ACTIVISTS ----------------
    elif agent.type == ATypes.ACTIVIST:
        #  institution_type, based on this, lets put emoji on the grid

        if agent.inProtest:
            color = "red"
        elif agent.jailTerm:
            color = "lightpink"
        else:
            color = "lightpink"

        marker = "." if agent.jailTerm else "o"

    # ---------------- POLICE ----------------
    elif agent.type == ATypes.POLICE:
        # state: defected vs loyal
        color = "lightgray" if agent.defect else "black"

        # trait: defection propensity
        if agent.defectLevel == "high":
            size = 30
            # size = 100
        else:
            size = 30

        marker = "v"

    # ---------------- PILLARS ----------------
    elif agent.type == ATypes.PILLAR:
        color = "thistle" if agent.defect else "fuchsia"

        if agent.defectLevel == "high":
            size = 30
        else:
            size = 30
        marker = "*"

    return {"color": color, "size": size, "marker": marker}


# Expanded to include research-oriented markers
PILLAR_METADATA = {
    "Educational System": {"emoji": "🎓", "tag": "EDU"},
    "Trade Unions": {"emoji": "✊", "tag": "LAB"},
    "Religious Institutions": {"emoji": "⛪", "tag": "REL"},
    "Professional Associations": {"emoji": "🛡️", "tag": "PROF"}, # Better: Protection of standards
    "Independent Media": {"emoji": "🔍", "tag": "WATCH"},       # Better: The 'Watchdog'
    "Business Community": {"emoji": "💼", "tag": "ECON"},       # Better: Professional/Capital
    "Civil Service": {"emoji": "⚙️", "tag": "EXEC"},           # Better: The 'Machine' of state
    "Local Government": {"emoji": "📍", "tag": "SUB-NAT"},    # Better: Urban/Regional authority
    "Parliament": {"emoji": "🏛️", "tag": "LEG"},
    "Supreme Court": {"emoji": "⚖️", "tag": "JUD"},
    "State Media": {"emoji": "📢", "tag": "PROP"},             # Sourced/State-controlled
}

In [5]:
# Expanded to include research-oriented markers
PILLAR_METADATA = {
    "Educational System": {"emoji": "🎓", "tag": "EDU"},
    "Trade Unions": {"emoji": "✊", "tag": "LAB"},
    "Religious Institutions": {"emoji": "⛪", "tag": "REL"},
    "Professional Associations": {"emoji": "🛡️", "tag": "PROF"}, # Better: Protection of standards
    "Independent Media": {"emoji": "🔍", "tag": "WATCH"},       # Better: The 'Watchdog'
    "Business Community": {"emoji": "💼", "tag": "ECON"},       # Better: Professional/Capital
    "Civil Service": {"emoji": "⚙️", "tag": "EXEC"},           # Better: The 'Machine' of state
    "Local Government": {"emoji": "📍", "tag": "SUB-NAT"},    # Better: Urban/Regional authority
    "Parliament": {"emoji": "🏛️", "tag": "LEG"},
    "Supreme Court": {"emoji": "⚖️", "tag": "JUD"},
    "State Media": {"emoji": "📢", "tag": "PROP"},             # Sourced/State-controlled
}


PILLAR_METADATA = {
    "Educational System":       {"emoji": "🎓", "tag": "EDU"},
    "Trade Unions":             {"emoji": "✊", "tag": "LAB"},
    "Religious Institutions":   {"emoji": "⛪", "tag": "REL"},
    "Professional Associations":{"emoji": "🛡️","tag": "PRO"},
    "Independent Media":        {"emoji": "🔍", "tag": "WDG"},
    "Business Community":       {"emoji": "💼", "tag": "ECO"},
    "Civil Service":            {"emoji": "⚙️", "tag": "EXC"},
    "Local Government":         {"emoji": "📍", "tag": "LOC"},
    "Parliament":               {"emoji": "🏛️", "tag": "LEG"},
    "Supreme Court":            {"emoji": "⚖️", "tag": "JUD"},
    "State Media":              {"emoji": "📢", "tag": "PRO"},
}


# Replace your emoji field with a "symbol" field
# PILLAR_METADATA = {
#     "Educational System":        {"symbol": "✎", "tag": "EDU"},   # U+270E pencil
#     "Trade Unions":              {"symbol": "⚒", "tag": "LAB"},   # U+2692 hammer & pick
#     "Religious Institutions":    {"symbol": "✚", "tag": "REL"},   # U+271A bold cross
#     "Professional Associations": {"symbol": "◈", "tag": "PROF"},  # U+25C8 diamond
#     "Independent Media":         {"symbol": "◉", "tag": "WATCH"}, # U+25C9 fisheye
#     "Business Community":        {"symbol": "◆", "tag": "ECON"},  # U+25C6 filled diamond
#     "Civil Service":             {"symbol": "⚙", "tag": "EXEC"},  # U+2699 gear (plain, not emoji variant)
#     "Local Government":          {"symbol": "⌂", "tag": "SUB"},   # U+2302 house
#     "Parliament":                {"symbol": "⊞", "tag": "LEG"},   # U+229E grid square
#     "Supreme Court":             {"symbol": "⚖", "tag": "JUD"},   # U+2696 scales (plain)
#     "State Media":               {"symbol": "⊛", "tag": "PROP"},  # U+229B circled asterisk
# }


PILLAR_METADATA = {
    # Symbol chosen for instant recognition
    "Educational System":        {"symbol": "✎", "tag": "EDU"},   # U+270F  pencil = learning/writing
    "Trade Unions":              {"symbol": "⚒", "tag": "LAB"},   # U+2692  hammer & pick = labor
    "Religious Institutions":    {"symbol": "✝", "tag": "REL"},   # U+271D  cross = religion (most universal)
    "Professional Associations": {"symbol": "★", "tag": "PROF"},  # U+2605  star = badge of standards
    "Independent Media":         {"symbol": "◎", "tag": "WATCH"}, # U+25CE  bullseye = the watchdog 'eye'
    "Business Community":        {"symbol": "$", "tag": "ECON"},  # U+0024  dollar = capital/economy
    "Civil Service":             {"symbol": "⚙", "tag": "EXEC"},  # U+2699  gear = machinery of state
    "Local Government":          {"symbol": "⌂", "tag": "SUB"},   # U+2302  house = local/home
    "Parliament":                {"symbol": "☰", "tag": "LEG"},   # U+2630  three lines = law document
    "Supreme Court":             {"symbol": "⚖", "tag": "JUD"},   # U+2696  scales = justice
    "State Media":               {"symbol": "▶", "tag": "PROP"},  # U+25B6  play button = broadcast
}

def agent_portrayal(agent):
    """Return color, size, marker, and text for Matplotlib rendering."""

    # Defaults
    color = "gray"
    marker = "o"
    size = 20    # Default scatter size
    text = ""    # Default no emoji

    # ---------------- CITIZENS ----------------
    if agent.type == ATypes.CITIZEN:
        marker = "o"
        size = 15  # Shrunk slightly to reduce visual noise
        
        if agent.jailTerm:
            color = "white"
        elif agent.inProtest:
            color = "cyan"
        elif agent.activeNV:
            color = "mediumaquamarine"
        elif agent.grievanceLevel == "high":
            color = "lightblue"
        else:
            color = "lightcyan"

    # ---------------- POLICE ----------------
    elif agent.type == ATypes.POLICE:
        color = "lightgray" if agent.defect else "black"
        marker = "v"
        size = 25
        # Optional: uncomment if you want police to be emojis too
        # text = "👮" if not agent.defect else "🕊️"
        # size = 12

    # ---------------- ACTIVISTS ----------------
    elif agent.type == ATypes.ACTIVIST:
        color = "red" if agent.inProtest else "lightpink"
        
        if agent.jailTerm:
            marker = "."
            size = 10
        else:
            if hasattr(agent, "target_institution"):
                text = PILLAR_METADATA.get(agent.target_institution, {}).get("symbol", "?")
                size = 9   # fontsize in points — readable but smaller than pillars
            else:
                marker = "o"
                size = 30

    # ---------------- PILLARS ----------------
    elif agent.type == ATypes.PILLAR:
        color = "thistle" if agent.defect else "fuchsia"
        # color = "mediumpurple" if agent.defect else "royalblue"  # ← uncomment & pick colors
        
        if hasattr(agent, "institution_type"):
            text = PILLAR_METADATA.get(agent.institution_type, {}).get("symbol", "?")
            size = 15 # fontsize in points — was 5, which is nearly invisible
        else:
            marker = "*"
            size = 40
            
    return {"color": color, "size": size, "marker": marker, "text": text}

In [6]:
print("citizens", 1600*0.7)
print("police", 1600*0.04)
print("activists", 1600*0.008)
print("pillar", 1600*0.0153)
1133+84
# 64+24

citizens 1120.0
police 64.0
activists 12.8
pillar 24.48


1217

In [7]:
import glob, dill

save_dir = 'NVResistance_LLMResults/'
# 2 conflicting paramters PctPillar (1.53 vs 0.85), ProtestCycle (8.9 vs 7)
model_params = {   # values from S1 table, chenoweth initial optimized values
    'MaxSteps':200, 'PctCitizen':70, 'PctActivist':0.8, 'PctPolice':4, 'PctPillar':1.53, 'MaxJailTerm':10,
    'StartLegitimacy':0.56, 'PctFindNV':40, 'PctTargetNV':25, 'PctKillNV':10, 'BackfireCoeff':0.99,
    'valueF':0.0706, 'ProtestCycle':8.9, 'ProtestDuration':1, 'nNVthreshold':1, 'DefectThresh':0.05,
    'PeerPressureNum':3.46, 'DelayStartMax':5, 'NVSuccessPct':10, 'PillarProxStrat':3,
    'DefectThreshStDev':0.01, 'ActivistSearchVision':10, 'PctFickle':100, 'PctImmedProtest':5,
    'dataCollectorType':displayMode
}


# overrides model_params, put sliders here
param_visual_overrides = {
    "NVSuccessPct":{ "type": "SliderFloat", "value": model_params['NVSuccessPct'], "label": "NV Win % (Pillars)", "min": 5, "max": 50, "step": 1}
}


if displayMode == "singleRunVisualize":
    for key in param_visual_overrides.keys():
        model_params[key] = param_visual_overrides[key]
    page = JupyterViz(
        NVResistanceModel,
        model_params,
        measures = [
            {
             "protest":"cyan",
             "activists":"red",
             "police":"black", 
             "pillars":"fuchsia", 
            #  "activeNV":"mediumaquamarine"
             },
            make_text(lambda x: f"### Model Seed: {x._seed}"),
        ],
        name = "",
        agent_portrayal = agent_portrayal,
        play_interval=0,
    )
elif displayMode == "batchRun":
    
    total_iterations = 10
    chunk_size = 2
    save_folder = "NVResistance_BatchResult_actvt"

    os.makedirs(save_folder, exist_ok=True)

    for start in range(0, total_iterations, chunk_size):
        end = start + chunk_size

        print(f"\n=== Running iterations {start} to {end} ===")

        results = batchrunner.batch_run(
            NVResistanceModel,
            model_params,
            number_processes=None,                     # keep None for multiprocessing
            iterations=chunk_size,             # number of runs in this chunk
            max_steps=model_params["MaxSteps"],
            display_progress=True
        )

        filename = f"multi_run_data_{start}-{end}.db"
        save_path = os.path.join(save_folder, filename)

        with open(save_path, "wb") as f:
            dill.dump(results, f)

        print(f"Saved → {save_path}")

elif displayMode == "loadSession":
    dill.load_session('multi_run_data.db')
    page = "Loaded Runs"

page



Cannot show ipywidgets in text

In [18]:
# select = 1   # batchRun
# displayMode = ["singleRunVisualize", "batchRun", "loadSession"][select]

# orchestrator = RunOrchestrator(
#     base_config  = LogConfig(
#         provider    = "huggingface_llama",
#         prompt_type = "ZeroShot",
#         memory      = "stateful",
#         vision      = "vision-4",
#         base_dir    = "SimulationLogs",
#     ),
#     params       = model_params,
#     max_workers  = 1,    # must be 1 — HuggingFace loads model in main process
#     start_run_id = 0,
# )

# summary = orchestrator.run(n_runs=3)   # start with 3 to verify it works
# # ```

# # ---

# # ## Step 6 — First run will download the model

# # The first time you run, HuggingFace downloads ~16GB of weights to your cache. This takes 10-20 minutes depending on your internet. Subsequent runs use the cached version and start in ~30 seconds.

# # To control where the model is cached, add to your `.env`:
# # ```
# # HF_HOME=C:/Users/fmahnoor1/.cache/huggingface